# Replay a signal through a measured underwater acoustic channel

This notebook takes a signal of your own design, drives it through a channel that was
really measured at sea, adds ocean noise with measured statistics, and looks at what
comes back.

You supply the channel files. Download them once from Zenodo,
[doi:10.5281/zenodo.21287414](https://doi.org/10.5281/zenodo.21287414), and then:

- **In your browser**, pick them with the button in section 2. They are read straight
  off your disk by the browser's own file dialog. Nothing is uploaded anywhere.
- **On your laptop**, put them in the same folder as this notebook and the button is
  not needed.

The example below assumes `blue_1`, a 13 kHz mobile channel recorded in the North
Atlantic during the MACE'10 experiment, together with its noise file `blue_noise.mat`.
Any channel from the record works: the code picks whichever files you give it.

## 1. Install the toolbox

In the browser this pulls the wheel from PyPI with `micropip`, together with NumPy,
SciPy, Matplotlib, h5py and ipywidgets. On a laptop it calls `pip`. Either way it
takes a moment the first time.

In [ ]:
import sys

if sys.platform == "emscripten":  # running inside the browser
    import micropip

    await micropip.install(["uwa-channels", "ipywidgets"])
else:
    import subprocess

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "uwa-channels", "ipywidgets"],
        check=True,
    )

import numpy as np
import scipy.signal as sg
import h5py
import matplotlib.pyplot as plt
import ipywidgets as widgets
from uwa_channels import replay, noisegen

%matplotlib inline
print("toolbox ready")

## 2. Choose your files

Run the next cell and a **Choose .mat files** button appears. Click it, then select
both the channel file and its noise file, for example `blue_1.mat` and
`blue_noise.mat`. You can select both at once.

The browser reads them from your disk and hands the bytes to Python. There is no
server in this page, so nothing leaves your machine. The word "upload" that the
button uses is the browser's own wording for choosing a file.

> **A 206 MB channel may be too much for a tab.** The whole file passes through the
> page and then into the kernel, on top of the Python runtime already loaded. If the
> tab dies, that is why, and the answer is to run this notebook on your laptop under
> Jupyter instead. Try the small noise file first to confirm the button works.

In [ ]:
picker = widgets.FileUpload(
    accept=".mat",
    multiple=True,
    description="Choose .mat files",
    layout=widgets.Layout(width="260px"),
)
picker

Once the button reports the files are in, run this cell. It writes them where the
kernel can read them quickly and works out which is the channel and which is the
noise.

In the browser the files go to `/tmp`, which is ordinary memory inside the kernel.
That matters: JupyterLite's own working directory is a drive backed by the service
worker, and `h5py` reads a channel in many small pieces, which across that boundary
is slow enough to look broken. On a laptop the files are read where they already are.

In [ ]:
import os


def local_path(name):
    """Where a file should live so that h5py can read it quickly."""
    return f"/tmp/{name}" if sys.platform == "emscripten" else name


def take(picker):
    """Save whatever was chosen, and sort it into channel and noise."""
    paths = {}

    for item in picker.value:
        name = item["name"]
        data = item["content"]  # bytes handed over by the browser
        path = local_path(name)
        with open(path, "wb") as f:
            f.write(data)
        paths[name] = path
        print(f"{name}: {len(data) / 1e6:.1f} MB ready at {path}")

    if not paths:  # nothing chosen: fall back to files sitting next to the notebook
        for name in os.listdir("."):
            if name.endswith(".mat"):
                paths[name] = name
                print(f"{name}: found beside the notebook")

    if not paths:
        raise RuntimeError(
            "No .mat files. Click the button above and choose a channel file and a "
            "noise file, or put them in this folder."
        )

    noise = [p for n, p in paths.items() if "noise" in n.lower()]
    channel = [p for n, p in paths.items() if "noise" not in n.lower()]

    if not channel:
        raise RuntimeError(f"Only noise files were given: {sorted(paths)}")
    if not noise:
        raise RuntimeError(f"No noise file among {sorted(paths)}")

    return channel[0], noise[0]


channel_path, noise_path = take(picker)
print(f"\nchannel: {channel_path}\nnoise:   {noise_path}")

## 3. Set the parameters

`fs` is the sampling rate we work at, `fc` the carrier frequency, and `R` the symbol
rate. `array_index` picks which hydrophones of the receiving array to use: here the
first, third, and fifth.

In [ ]:
fs = 48e3  # sampling rate [Hz]
fc = 13e3  # carrier frequency [Hz]
R = 4e3  # symbol rate [Hz]
n_repeat = 10  # how many times to repeat the symbol block
array_index = np.array([0, 2, 4])  # which receive elements to keep
textbook_noise = False  # True gives plain pink noise instead of measured statistics

## 4. Build the transmitted signal

A BPSK block, upsampled to the working rate and mixed up to the carrier. The silence
padded on each end leaves room for the channel delay spread, so nothing important
falls off the start or the end.

In [ ]:
data_symbols = np.random.choice([-1.0, +1.0], size=(1023,))
baseband = sg.resample_poly(np.tile(data_symbols, n_repeat), fs / R, 1)
passband = np.real(baseband * np.exp(2j * np.pi * fc * np.arange(len(baseband)) / fs))

input = np.concatenate((np.zeros((int(fs / 10),)), passband, np.zeros(int(fs / 10))))

print(f"transmitting {len(input) / fs:.2f} s at {fs / 1e3:.0f} kHz")

## 5. Replay it through the channel

This is the core call. It returns one column per element of `array_index`.

Pass `start=` to begin at a different point along the recording, which gives you a
different piece of the channel's evolution.

In the browser this cell is the slow one. Everything runs single-threaded in
WebAssembly, so where a laptop takes about two seconds, a tab takes several times
that, with one core pinned the whole while. That is normal. To make it quicker, go
back and set `n_repeat = 4` and `array_index = np.array([0])`.

In [ ]:
channel = h5py.File(channel_path, "r")
print("channel contents:", list(channel.keys()))

output = replay(input, fs, array_index, channel)
# output = replay(input, fs, array_index, channel, start=1000)

print("received signal shape:", output.shape)

## 6. Add ocean noise

`noisegen` draws noise with the statistics measured at the same site: the pink slope,
the correlation between array elements, and the heavy tails where the ambient field is
impulsive. Setting `textbook_noise = True` above replaces all of that with plain pink
Gaussian noise, which is a useful thing to compare against.

In [ ]:
noise = h5py.File(noise_path, "r")

if textbook_noise:
    output += 0.05 * noisegen(output.shape, fs)
else:
    output += 0.05 * noisegen(output.shape, fs, array_index, noise)

## 7. Downconvert to baseband

Mixing back down by the carrier leaves the complex baseband signal, which is what the
correlation below works on.

In [ ]:
v = output * np.exp(-2j * np.pi * fc * np.arange(output.shape[0])[:, None] / fs)

## 8. Look at what came back

### 8.1 The received waveform

One trace per hydrophone. The bursts of energy sit where the transmission was, and the
silence at each end is the padding we added.

In [ ]:
plt.figure()
plt.plot(np.arange(output.shape[0]) / fs, output)
plt.xlabel("Time [s]")
plt.ylabel("Received signal")
plt.title("Received signal at each element")
plt.show()

### 8.2 Cross-correlation

Correlating against a short piece of the transmitted block reveals the multipath
structure. Each peak is an arrival: the direct path first, then the surface and bottom
bounces behind it. This is the impulse response showing itself.

In [ ]:
plt.figure()
plt.plot(
    np.abs(
        np.correlate(v[:, 0], sg.resample_poly(data_symbols[:128], fs / R, 1), "full")
    )
)
plt.xlabel("Samples")
plt.ylabel("Xcorr")
plt.title("Cross-correlation at the first element")
plt.show()

### 8.3 Spectrum

The Welch estimate around the carrier, showing the transmitted band sitting on the
ocean noise floor.

This uses `scipy.signal.welch` rather than `plt.psd`. The two compute the same thing,
but `plt.psd` builds a sliding-window view spanning several gigabytes on paper, which
a 32-bit WebAssembly build refuses to create.

In [ ]:
# scipy's welch cuts the signal into segments. matplotlib's plt.psd instead builds a
# sliding-window view of the whole signal, about 133744 by 8192 here: harmless on a
# 64-bit machine, but wasm32 cannot express an array that nominally spans 8 GB, so it
# raises "array is too big" in the browser.
f_psd, pxx = sg.welch(output[:, 0], fs=fs, nperseg=8192)

plt.figure()
plt.plot(f_psd, 10 * np.log10(pxx))
plt.xlim(fc - R, fc + R)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Power/frequency [dB/Hz]")
plt.grid()
plt.title("Welch power spectral density estimate")
plt.show()

## Where to go next

- Change `array_index` to a single element, or to the whole array, and watch the
  correlation change.
- Move `start=` along the recording to hear a different stretch of the channel.
- Swap `blue_1.mat` for another channel from the
  [Zenodo record](https://doi.org/10.5281/zenodo.21287414). `black.mat` is the Mariana
  Trench, `red_1.mat` was recorded near Singapore.
- Set `textbook_noise = True` and compare the spectrum against the measured statistics.

The companion notebook for `unpack`, the documentation, and the MATLAB toolbox are all
linked from [uwa-channels.github.io](https://uwa-channels.github.io/).